<div class="alert alert-block alert-info">This Jupyter Notebook is part of the course <strong>"LLMs as Judges for Search" by OpenSource Connections.</strong>
    
Check out https://opensourceconnections.com/training/ for the full course and other classes.</div>


# Lab: Adding reasoning

In this lab we will add a user persona to our prompt. Please go ahead invent your own personas. We'll be adding some random prices to our data that you might want to reflect in your personas.


## Imports

In [2]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
import os
import openai
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score
import random


/opt/homebrew/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.25.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [5]:
load_dotenv(dotenv_path='../.env')

api_key = os.environ.get('OPENAI_API_KEY')

In [4]:
pd.set_option('display.max_colwidth', None)

## Product Data to Judge

Look at the following dataset. It contains search result data for the queries 'duct tape', 'iphone xr cool cases for teenage girls' and 'laptop'. This time we will only use products for the query ''iphone xr cool cases for teenage girls'.



In [6]:
df_products = pd.read_json('../data/esci-mini.json')

df_products

,query,product_id,esci_label,binary_label,label,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,duct tape,B00DJWTGAG,E,1,3,"3M 2979 Multi-Use Duct Tape, Silver, 1.88 in x 60 yd x 7 mil, 1 Pack, Temporary Repair, Patching, Tabbing, Capping Pipe, Marking, Labeling",None,"Commercial grade – Silver duct tape resists curling and tears off roll cleanly for light use in professional MRO/construction applications\nFlexible adhesive – Aggressive synthetic rubber adhesive sticks immediately to a wide variety of surfaces\nMulti-use applications– Great for patching, tabbing, light duty bundling, capping pipe, marking, labeling or temporary repair\nPriced for economy – This multipurpose contractor-grade tape is economically priced to best provide a quality temporary solution for light duties",3M,Silver,us
1,duct tape,B078M21NYH,E,1,3,"Craftzilla Rainbow Colored Duct Tape — 6 Bright Colors — 10 Yards x 2 Inch — No Residue, Tear by Hand & Waterproof — Great for Arts & Crafts, Color-Coding, and DIY Projects","This multi purpose rainbow set of duct tape colors and patterns is great for Students, Teachers, Parents, Artists, and Professionals. Use this colored duct tape for crafts and art projects that can be done in the classroom, studio, kitchen, home, or garage. The colorful duct tape rips nicely and cleanly so it can easily be used and applied to different surfaces. This fun duct tape bulk pack comes with 6 fun duct tape craft rolls - each roll measuring 10 yards of 2 inch duct tape. This includes 6 rolls in an assortment of bright neon duct tape colors (not fluorescent) including: Pink, Orange, Yellow, Green, Blue, Purple. This colorful tape set can be made into DIY arts and crafts projects such as the construction of forts, making duct tape wallets, keychains, origami, personalizing and decorating journals, notebooks, and luggages. This is also great for color-coding, labeling, and organizing use making it a fun addition to your DIY kits. Whenever you move or need storage organization, use these duct tape assorted colors to make labels for boxes and storage bins and put corresponding duct tape rainbow color tags to each room. Discover creative possibilities with these duct tape arts and crafts kits. Art is limitless so use this duct tape for boys, girls, adults, artists, and professionals!","Vivid Vibrance – Enjoy eye-catching brightness and a full rainbow of color options. Make labels, gifts, and statements—your multi purpose color duct tape pack from Craftzilla is both your palette and canvas.\nTearable and Easy to Clean – Assemble rainbow duct tape kits at home, in school, and on vacation. Your craft tapes are easy for tiny hands to tear for their craft and construction projects, and no hassle to peel off for easy cleanup with no residue.\nYou’re on a Roll – And you’ve got plenty left! With 60 total yards of wonderful hues and easy-tear workability, you can use your colored duct tape variety pack for project after project.\nCommunicate With Color – Apply your multi color duct tape anywhere you need to be heard without saying a word. Effortlessly label moving boxes, organize unruly TV cabling, and mark social-distancing spots.\nStick With Us – Count on Craftzilla for colored tape duct so fun, bright, and inspiring you won’t want to put it down. Your tape set is backed by our commitment to your colorful and crafty success.",Craftzilla,"Rainbow - Pink, Orange, Yellow, Green, Blue, Violet",us
2,duct tape,B0021L9MVO,I,0,0,"Duck HD Clear Heavy Duty Packing Tape, 1.88 Inch x 109 Yards, 6 Rolls (299016)",None,"Heavy duty for a strong and secure hold to keep your valuables safe while moving, shipping or in storage\nOffers wide temperature range performance for shipping and storage in hot or cold temperatures\nAdhesive bond strengthens over time for a long-lasting hold on boxes, perfect for storage\nCrystal clear to the core for a professional look on boxes or taping address labels\nMeets postal

Only use products for the query 'iphone xr cool cases for teenage girls'.

Add some random prices in the range between 20 and 60 USD.

In [9]:
random.seed(42)
df_cases = df_products[df_products['query'] == 'iphone xr cool cases for teenage girls'].copy()
df_cases['price'] = [random.randint(20, 60) for _ in range(df_cases.shape[0])]
df_cases

,query,product_id,esci_label,binary_label,label,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,price
10,iphone xr cool cases for teenage girls,B07VN8CM9D,E,1,3,"iPhone XR Case,Red Rose Star iPhone XR Cases for Girls,Tempered Glass Back Cover Anti Scratch Reinforced Corners Soft TPU Bumper Shockproof Case for iPhone XR Starstruck Flower Color","<p><b>drophead is a company that specializes in selling mobile phone accessories, such as the pattern of toughened glass mobile phone case, which is carefully designed by us. The accessories are deeply rooted in creativity, quality and style.</b></p> <p>If you have suggestions for our products and services, you can inform us and leave feedback for our customer service.</p> <p><b>About the drophead iPhone XR Case<p><b> <ul> <li>iPhone XR Case is made of glass back cover and soft silicone TPU rubber around it.</li> </ul> <ul> <li>iPhone XR Case Protecting your device from scratches, dust, shock and fingerprint.</li> </ul> <ul> <li>High Quality Material to use for a longer time.Provided great protection for your iPhone XR.</li> </ul> <ul> <li>The hole position of iPhone XR Case is accurate and correct.</li> </ul> <ul> <li>Raised lips ensure extra protection of screen and back camera.Protecting your device from scratches, dust, shock and fingerprint</li> </ul> <ul> <li>Fits perfectly,full access to all ports, buttons, and features, Precise cutout ensures full access to all the functions and ports for your phone.</li> </ul> <ul> <li>Reinforcement and protection of the four corners of iPhone XR Case can prevent your mobile phone from breaking down and protect your mobile phone more durably.</li> </ul>","The company uses international advanced equipment and ink, clear patterns, bright colors, never fade. Your iPhone XR case is made of flexible thermoplastic polyurethane (TPU) and toughened glass.The personalized design of glass rear case makes your mobile phone more beautiful, scratch-resistant and wear-resistant,light and comfortable.\nPrecise cutout ensures full access to all the functions and ports for your iPhone XR; Sensitive button covers allow quick responsiveness.\nFULL-EDGE PROTECTION:Raised screen edge design and reinforced corner bumpers provide thorough security for your slim device at all sides and angles. The sleek raised-lip protects features of your iPhone XR – such as the screen, camera, and back surface – from scratches when lying flat.\nThe case of iPhone XR has four-corner reinforcement protection function, which can prevent the mobile phone from being strongly impacted when accidentally sliding, and more effectively protect the mobile phone.\n180 days 100% Money Back Guarantee:If you have any quality issues or any questions about your iPhone XR case, please feel free to contact us at the first time. We promise to resend you a new phone case.",ZHEGAILIAN,Red Rose Star,us,60
11,iphone xr cool cases for teenage girls,B07BGXBD1C,I,0,0,"A-Focus Compatible with iPhone 8 Plus Case for Girls, iPhone 7 Plus Case Pink, Colorful Pink Blue Red Abstract Cloud Frosted Shock Proof Slim Shell Case for iPhone 7 Plus 8 Plus 5.5 inch Matte Pink 4","1.Compatible with Apple iPhone 7 Plus (2016) & iPhone 8 Plus (2017).<br>2.IMD (In-Mould-Decoration) Technology: HD Color Printed under a Layer of PET, this print will never come off or fade.<br>3.Matte Surface, made of soft flexible TPU, full print, look like marble .<br>4.Full and easy access to the charging port, headphone jack, silent button, volume buttons and silent button, your device stays fully functional when it's in the case.<br>5.Package include: 1 x A-Focus Case for Apple iPhone 7 / 8 plus 5.5"", Accessory ONLY, Phone not included.","Compatible with Apple iPhone 7 Plus (2016) & iPhone 8 Plus (2017)\nIMD (In-Mould-Decoration) Technology: HD Color Printed under a Layer of PET, this print will never come off or fade\nMatte Surface, made of soft flexible TPU, full print, look like marble\nF

In [10]:
SYSTEM_PROMPT = """
You are an expert relevance judgment system. Your task is to assess the relevance of a given document to a specific user query.
The document has a product title and a price in USD. You will also be supplied with some information about the user.

Provide a relevance rating by assigning the label 'relevant' or 'not relevant', given the query, the user persona and the product.

In addition to the rating, provide a concise reasoning for your judgment.
First reason, then judge based on the reasoning.

Provide your relevance rating by assigning the label 'relevant' or 'not relevant' as property 'relevance' of an object in JSON format and the reasoning as property 'reasoning'.
"""

In [11]:
def make_user_prompt(query, doc_title, persona, price):
    return f"""
    
User Query: {query}

User Persona: {persona}

Document title:\n{doc_title}

Price: {price}

Based on the above, provide a relevance judgment."""



In [12]:
CLIENT = OpenAI(api_key=api_key)

def evaluate(query, doc_id, doc_title, persona, price, response_judgment_property='relevance', response_reasoning_property='reasoning', client=CLIENT, system_prompt=SYSTEM_PROMPT):
    
    try:
        # Send request to OpenAI API
        # Using generate_content and specifying the response_mime_type for JSON output
       
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": system_prompt},
                {
                    "role": "user",
                    "content": make_user_prompt(query, doc_title, persona, price),
                },
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        
        # Parse the JSON response
        json_output = response.choices[0].message.content
        judgment = json.loads(json_output)
        #print(judgment)
        
        return judgment[response_judgment_property], judgment[response_reasoning_property]
        
        

    except Exception as e:
        print(f"Error processing OpenAI judgment for Query: '{query[:30]}...', Doc ID: {doc_id}: {e}")
        raise e
        #return None
        

In [13]:
def evaluate_dataset(df, persona, response_judgment_property='relevance', response_reasoning_property='reasoning', client=CLIENT, system_prompt=SYSTEM_PROMPT):
    df = df.copy()
    df[['judgment','reasoning']] = df.apply(lambda row: evaluate(query=row['query'], doc_id=row['product_id'], doc_title=row['product_title'], 
                                                                 price=row['price'], persona=persona,
                                    response_judgment_property=response_judgment_property, 
                                    response_reasoning_property=response_reasoning_property, 
                                    client=client, system_prompt=system_prompt), axis=1, result_type="expand")
    df['binary_judgment'] = df['judgment'].apply(lambda j: 1 if j == 'relevant' else 0)

    return df
    

    

Define your persons below:

In [14]:
PERSONA = "A 50 year old person who always buys the best products and is shopping for their grand-daughter"

In [15]:
df_eval = evaluate_dataset(df_cases, persona=PERSONA)

In [16]:
df_eval[['query', 'product_title', 'price', 'judgment', 'reasoning']]

,query,product_title,price,judgment,reasoning
10,iphone xr cool cases for teenage girls,"iPhone XR Case,Red Rose Star iPhone XR Cases for Girls,Tempered Glass Back Cover Anti Scratch Reinforced Corners Soft TPU Bumper Shockproof Case for iPhone XR Starstruck Flower Color",60,relevant,"The document features an iPhone XR case specifically designed for girls, which aligns with the user query for 'cool cases for teenage girls.' The product title emphasizes its appeal to girls with the design and features, making it suitable for the user's granddaughter. The price indicates a premium product, which matches the user's persona of buying the best products."
11,iphone xr cool cases for teenage girls,"A-Focus Compatible with iPhone 8 Plus Case for Girls, iPhone 7 Plus Case Pink, Colorful Pink Blue Red Abstract Cloud Frosted Shock Proof Slim Shell Case for iPhone 7 Plus 8 Plus 5.5 inch Matte Pink 4",27,not relevant,"The document features a case for iPhone 7 Plus and 8 Plus, which does not match the user's query for iPhone XR cases. Additionally, while it is marketed towards girls, the specific model mismatch makes it irrelevant for the user's intent to find a cool case for their granddaughter's iPhone XR."
12,iphone xr cool cases for teenage girls,"Artbling Case for iPhone XR 6.1"" Silicone 3D Cartoon Animal Cover,Kids Girls Boys Cool Cute Gost Cases,Kawaii Soft Gel Rubber Unique Fun Character Fashion Funny Protector for iPhoneXR (Green Alien)",21,not relevant,"The document features a case for the iPhone XR that is described as a '3D Cartoon Animal Cover' targeted towards kids and includes terms like 'Kawaii' and 'Cute', which may not align with the preferences of a teenage girl. Additionally, the user persona is a 50-year-old person shopping for their granddaughter, who may be looking for more stylish or trendy options rather than something that is explicitly marketed towards younger children."
13,iphone xr cool cases for teenage girls,"iPhone Xs Case for Girls, YeLoveHaw Flexible Soft Slim Fit Full-Around Protective Cute Shell Phone Case Cover with Purple Floral and Gray Leaves Pattern for iPhone X/XS 5.8 Inch (Pink Flowers)",37,not relevant,"The document features a case for the iPhone Xs, not the iPhone XR as specified in the user query. Additionally, the user is a 50-year-old person shopping for their granddaughter, which suggests they may be looking for something specifically appealing to teenage girls. While the case is described as cute, it does not match the specific model requested, making it not relevant."
14,iphone xr cool cases for teenage girls,"ooooops iPhone 8 Case,7 Case,SE Case for Girls, Green Leaves with White&Brown Flower Pattern Design, Slim Fit Clear Bumper Soft Full-Body Protective Cover Case for iPhone 7/8/SE 4.7''(Leaves&Flowers)",35,not relevant,"The document features a case for iPhone 7/8/SE, which does not match the user's query for iPhone XR cases. Additionally, while the design may appeal to teenage girls, the user is specifically looking for cases for an iPhone XR, making this product irrelevant to their needs."
15,iphone xr cool cases for teenage girls,"Ruky Case for iPhone XR Glitter Case, Gradient Quicksand Series TPU Bumper Cushion Reinforced Corners Protective Bling Liquid Girls Women Case for iPhone XR 6.1 inches, Gradient Pink",34,relevant,"The document features a case specifically designed for the iPhone XR, which is the user's query. The product is described as a glitter case with a gradient pink design, appealing to teenage girls. Although the user is a 50-year-old shopping for their granddaughter, the product aligns well with the preferences of a teenage girl, making it a suitable choice."
16,iphone xr cool cases for teenage girls,"Coralogo for iPhone XR TPU Case, 3D Cute Cartoon Funny Design Unique Character Protective Kawaii Fashion Fun Cool Stylish Cover Kits Skin Teens Kids Girls Boys Cases for iPhone XR 6.1"" (Sponge Patrick",28,relevant,"The document features a case specifically designed fo